# Institution Effects

Reproduces the institution analyses in the Appendix (Institution Effects):

1. **Within-cancer-type TSS enrichment of the probe-score extreme pool** — summed within-type Cramér's V against a label-stratified permutation null (`experiments/tcga_validation/run_tss_enrichment.py`).
2. **Pan-cancer within-TSS AUROC collapse** — Table `tab:app-tss`. The probe AUROC is stratified by tissue source site (TSS) instead of cancer type.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path('../../data')
OUT_DIR = DATA_DIR / 'tcga'


## Within-cancer-type TSS enrichment of the probe-score extreme pool

Records from `experiments/tcga_validation/run_tss_enrichment.py`. For each FM × label × pooled random split, a probe trained on A′ scores the held-out patients (B ∪ C), and the extreme pool E is the top 10% by probe score. Within each cancer type with enough patients, Cramér's V compares the TSS mix of E with that of the other patients of the same type; V is summed over cancer types (independent patients) and compared with a null in which E is permuted within label strata, so enrichment explained by differing label prevalence across sites is removed (1,000 permutations).

Splits share most patients, so p-values are summarized per FM × label by the median over splits rather than combined across splits or FMs.

In [ ]:
recs = pd.read_csv(OUT_DIR / 'tss_enrichment_records.csv')
summary = (recs.groupby(['label', 'fm'])
           .agg(n_ct_min=('n_ct', 'min'), n_ct_max=('n_ct', 'max'),
                sum_v=('sum_v', 'mean'), null95=('null95', 'mean'),
                p_median=('p', 'median'), frac_p05=('p', lambda x: (x < 0.05).mean()))
           .reset_index())
print(summary.round(4).to_string(index=False))
print(summary.groupby('label').p_median.agg(['min', 'max']).round(4))

## Pan-cancer within-TSS AUROC collapse

For each FM × label and each of the 50 pooled random splits, a probe trained on A′ scores the held-out patients (B ∪ C). The pan-cancer AUROC is compared with the pair-weighted mean AUROC within TSS strata that have at least 5 positive and 5 negative patients.

The computation needs the raw FM embeddings, which are not redistributed. If `pan_cancer_within_tss_auroc_collapse.csv` is present in `OUT_DIR` it is loaded; otherwise it is recomputed from the embeddings in `FMB_DIR`.

In [ ]:
import json

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

COLLAPSE_CSV = OUT_DIR / 'pan_cancer_within_tss_auroc_collapse.csv'
FMB_DIR = DATA_DIR / 'embeddings/pathology'  # raw FM embeddings, not in data.tar.xz
FM_PATHS = {
    'TITAN':        FMB_DIR / 'titan/titan_20x.csv',
    'CONCH':        FMB_DIR / 'conch/conch1_5_20x_MEAN.csv',
    'UNI':          FMB_DIR / 'uni/univ2_20x_MEAN.csv',
    'H-Optimus-0':  FMB_DIR / 'hoptimus/hoptimus0_20x_MEAN.csv',
    'OpenMidnight': FMB_DIR / 'openmidnight/openmidnight_20x_MEAN.csv',
}
LABEL_ETA2 = {'tmb': 0.44, 'tp53': 0.34, 'age': 0.27}


def compute_pan_vs_within_tss(Z_all, y, tss, splits, min_per_class=5):
    """Compute pan-cohort AUROC and within-TSS pair-weighted C-statistic across splits.
    
    Mirrors Fig 4c logic: same probe, same held-out patients, different stratification.
    """
    pan_aurocs, within_aurocs = [], []

    for sp in splits:
        ap_idx  = np.array(sp['cohort_aprime'])
        held_idx = np.concatenate([np.array(sp['cohort_b']), np.array(sp['cohort_c'])])

        # Filter to valid (have embedding + label)
        ap_valid = ap_idx[(y[ap_idx] >= 0) & ~np.isnan(Z_all[ap_idx, 0])]
        held_valid = held_idx[(y[held_idx] >= 0) & ~np.isnan(Z_all[held_idx, 0])]
        if len(ap_valid) < 20 or len(held_valid) < 20:
            continue

        # Train probe on A'
        scaler = StandardScaler().fit(Z_all[ap_valid])
        Z_ap_s = scaler.transform(Z_all[ap_valid])
        y_ap = y[ap_valid]
        if (y_ap == 1).sum() < 2 or (y_ap == 0).sum() < 2:
            continue
        clf = LogisticRegression(C=0.1, max_iter=500, solver='liblinear')
        clf.fit(Z_ap_s, y_ap)

        # Pan-cohort AUROC on held-out
        Z_h = scaler.transform(Z_all[held_valid])
        y_h = y[held_valid]
        scores_h = clf.decision_function(Z_h)
        if (y_h == 1).sum() < 2 or (y_h == 0).sum() < 2:
            continue
        pan_auc = roc_auc_score(y_h, scores_h)

        # Within-TSS pair-weighted C-statistic
        tss_h = tss[held_valid]
        concordant_w, total_pairs = 0.0, 0.0
        for site in np.unique(tss_h):
            mask = tss_h == site
            yt, st = y_h[mask], scores_h[mask]
            n_pos, n_neg = int((yt == 1).sum()), int((yt == 0).sum())
            if n_pos < min_per_class or n_neg < min_per_class:
                continue
            auc = roc_auc_score(yt, st)
            n_pairs = n_pos * n_neg
            concordant_w += auc * n_pairs
            total_pairs += n_pairs

        if total_pairs == 0:
            continue
        within_auc = concordant_w / total_pairs
        pan_aurocs.append(pan_auc)
        within_aurocs.append(within_auc)

    if not pan_aurocs:
        return None
    pan_m = np.mean(pan_aurocs)
    wit_m = np.mean(within_aurocs)
    return {
        'pan_mean': pan_m, 'pan_se': np.std(pan_aurocs) / np.sqrt(len(pan_aurocs)),
        'within_tss_mean': wit_m, 'within_tss_se': np.std(within_aurocs) / np.sqrt(len(within_aurocs)),
        'delta': pan_m - wit_m,
        'n_splits': len(pan_aurocs),
    }


if COLLAPSE_CSV.exists():
    df_pan_collapse = pd.read_csv(COLLAPSE_CSV)
    print(f'Loaded {COLLAPSE_CSV}')
else:
    # Patient metadata (patient-level data, not in data.tar.xz; index-ordered, matches split indices)
    meta = pd.read_csv(OUT_DIR / 'patient_meta.csv')
    meta['tss'] = meta['patient_id'].str[5:7]
    pooled_splits = json.load(open(OUT_DIR / 'pooled_random_splits.json'))
    task_data = np.load(OUT_DIR / 'task_labels.npz')
    labels_all = {lt: task_data[f's2b_{lt}'] for lt in LABEL_ETA2}  # binarized; -1 = missing
    tss_all = meta['tss'].values
    collapse_rows = []
    for fm_name, fm_path in FM_PATHS.items():
        print(f'Loading {fm_name}...')
        df_fm = pd.read_csv(fm_path, low_memory=False)
        df_fm['patient_key'] = df_fm['USUBJID'].apply(
            lambda x: '-'.join(str(x).replace('TCGA/', '').split('-')[:3]))
        df_fm = df_fm.drop_duplicates('patient_key').set_index('patient_key')
        ecols = [c for c in df_fm.columns if c.startswith('E')]
        Z = np.full((len(meta), len(ecols)), np.nan, dtype=np.float32)
        for i, pid in enumerate(meta['patient_id']):
            if pid in df_fm.index:
                Z[i] = df_fm.loc[pid, ecols].values.astype(np.float32)
        n_valid = (~np.isnan(Z[:, 0])).sum()
        print(f'  {n_valid}/{len(meta)} patients with embeddings')

        for lt, y in labels_all.items():
            res = compute_pan_vs_within_tss(Z, y, tss_all, pooled_splits)
            if res:
                row = {'FM': fm_name, 'label': lt, 'eta2_cancer_type': LABEL_ETA2[lt], **res}
                collapse_rows.append(row)
                print(f'  {lt}: pan={res["pan_mean"]:.3f}, within-TSS={res["within_tss_mean"]:.3f}, Δ={res["delta"]:+.3f}')
    df_pan_collapse = pd.DataFrame(collapse_rows)
    df_pan_collapse.to_csv(COLLAPSE_CSV, index=False)

In [ ]:
FM_ORDER = ['TITAN', 'CONCH', 'UNI', 'H-Optimus-0', 'OpenMidnight']
table = (df_pan_collapse
         .assign(label=lambda d: d['label'].str.upper().replace({'AGE': 'Age'}))
         .pivot(index='FM', columns='label', values=['pan_mean', 'within_tss_mean', 'delta'])
         .rename(columns={'pan_mean': 'Pan', 'within_tss_mean': 'TSS', 'delta': 'Delta'})
         .swaplevel(axis=1)
         .reindex(columns=pd.MultiIndex.from_product([['TMB', 'TP53', 'Age'], ['Pan', 'TSS', 'Delta']]))
         .reindex(FM_ORDER))
print(table.to_string(float_format='{:.3f}'.format))